In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as cp
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 17
a = 0.2
N = 500
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_p = TDECalculator('MAMS1Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_p = mass_p.rel_whole_star_sample()

In [5]:
radii_p = sample_p['rr']

rtde = mass_p.R_TDE
Lz = mass_p.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_p <= 0.5,
    rtde - radii_p * mass_p.Rstar,
    rtde + radii_p * mass_p.Rstar
)

deltaE = mass_p.Rstar / mass_p.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_p['dEnergy_random'] * deltaE 
dLz = mass_p.dLz_random
dQ = mass_p.dQ_random
mass_ratio = mass_p.mass_ratio

In [6]:
print(f"Lz = {Lz}")
print(f"dLz range: [{dLz.min():.4e}, {dLz.max():.4e}]")
print(f"total_Lz range: [{(Lz + dLz).min():.4e}, {(Lz + dLz).max():.4e}]")
print(f"dE range: [{dE.min():.4e}, {dE.max():.4e}]")
print(f"total_E range: [{(E + dE).min():.4e}, {(E + dE).max():.4e}]")
print(f"bound fraction: {((E + dE) < 1.0).mean():.3f}")

Lz = 6.181342497129548
dLz range: [-1.3651e-01, 1.3650e-01]
total_Lz range: [6.0448e+00, 6.3178e+00]
dE range: [-1.2764e-03, 1.2764e-03]
total_E range: [9.9872e-01, 1.0013e+00]
bound fraction: 0.496


In [7]:
dT_p = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_p.npy', dT_p.dTs)
gc.collect()

Computing radial periods for 101,880,000 particles ...
  E  range: [0.998724, 1.001276]
  Q  range: [-4.519e+01, 4.519e+01]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 10115/10115 (100%)
  Bound: 50,573,859 / 101,880,000
  Valid roots: 46,409,587
  Valid Lambda_r: 46,409,587
  Quadrature: 929 chunks ...
    chunk 929/929  (100%)
  Successful T_r: 46,409,587 / 101,880,000


20

In [8]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample['dEnergy_random']
    dT_rand = dT / delta
    dMass   = whole_star_sample['dMass']

    bins_E = np.linspace(-2, 2, 1000)
    bins_T = np.logspace(0, 6, 1000)

    # dE: use all particles with finite energy and mass
    valid_E = np.isfinite(dE_rand)
    hist_E, edges_E = np.histogram(dE_rand[valid_E], bins=bins_E,
                                   weights=dMass[valid_E], density=True)

    # dT: only particles with valid finite period within bin range
    valid_T = (np.isfinite(dT_rand) & np.isfinite(dE_rand)
               & (dT_rand >= 1.0) & (dT_rand <= 1e6))
    hist_T, edges_T = np.histogram(dT_rand[valid_T], bins=bins_T,
                                   weights=dMass[valid_T], density=True)

    return {"x": 0.5*(edges_E[:-1]+edges_E[1:]), "y": hist_E}, \
           {"x": 0.5*(edges_T[:-1]+edges_T[1:]), "y": hist_T}

In [9]:
n_ex = TDECalculator('MAMS1Msun', Rp=Rp, a=0.0, N=N)
DeltaE = n_ex.Rstar / n_ex.Rp**2
DeltaT = 1 / DeltaE**1.5

In [10]:
rel_p_E, rel_p_T = make_plot_dicts(sample_p, dT_p.dTs, DeltaT)

In [11]:
import json

adden = "m1_rp17_a0p2"

with open(f"fallback_curves/rel_p_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_E.items()}, f)
with open(f"fallback_curves/rel_p_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_T.items()}, f)

In [12]:
print(f"dQ range: [{mass_p.dQ_random.min():.4e}, {mass_p.dQ_random.max():.4e}]")
print(f"dK range: [{mass_p.dK_random.min():.4e}, {mass_p.dK_random.max():.4e}]")
print(f"dLz range: [{mass_p.dLz_random.min():.4e}, {mass_p.dLz_random.max():.4e}]")
print(f"2*(Lz-a*E) = {2*(mass_p.mom_kerr_analytic(Rp,a) - a*1.0):.4f}")
cross_term = 2*(mass_p.mom_kerr_analytic(Rp,a) - a*1.0)*(mass_p.dLz_random - a*mass_p.dQ_random)
print(f"cross term range: [{cross_term.min():.4e}, {cross_term.max():.4e}]")
print(f"fraction Q < 0: {(mass_p.dQ_random < -mass_p.Carter).mean():.4f}")

dQ range: [-4.5190e+01, 4.5189e+01]
dK range: [-4.6803e+01, 4.6801e+01]
dLz range: [-1.3651e-01, 1.3650e-01]
2*(Lz-a*E) = 11.9627
cross term range: [-1.0650e+02, 1.0650e+02]
fraction Q < 0: 0.4963


In [13]:
idx_min_p = np.nanargmin(dT_p.dTs)
print(f"prograde min period particle - E: {dT_p.total_E.ravel()[idx_min_p]:.6f}, Lz: {dT_p.total_Lz.ravel()[idx_min_p]:.6f}, Q: {dT_p.total_Q.ravel()[idx_min_p]:.6f}")

prograde min period particle - E: 0.999312, Lz: 6.095710, Q: -23.907238


In [14]:
mass_p.dQ_random.max()

np.float64(45.189244852576124)

In [15]:
print(f"DeltaT: {DeltaT:.6f}")
print(f"nanmin dT_p.dTs: {np.nanmin(dT_p.dTs):.6f}")

DeltaT: 14410.513237
nanmin dT_p.dTs: 123532.090595
